# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print basic metadata
print(f"Dataset name: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'Unknown')}")
print(f"License: {getattr(metadata, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets and their fields using their `@id`s.

_Note: All entity references use their Croissant `@id`._

In [ ]:
# List all available record sets with their @id and names
record_sets = list(dataset.record_sets)
print(f"Total record sets in the dataset: {len(record_sets)}\n")

for rs in record_sets:
    print(f"RecordSet @id: {rs.id} | name: {getattr(rs, 'name', 'N/A')}")
    # Show the fields contained in each record set by their @id
    print("  Fields (with @id):")
    for f in rs.fields:
        print(f"    - {f.id} (label: {getattr(f, 'label', f.id)}) [dataType: {getattr(f, 'dataType', 'N/A')}]" )
    print()

# If record_sets is empty, show a troubleshooting message
if len(record_sets) == 0:
    print("Warning: No record sets were found in this dataset. Check the dataset schema or contact the data provider.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we load the first (or main) record set and print its DataFrame's columns using Croissant `@id` convention.

In [ ]:
# If there are record sets in the dataset, extract by their @id
dataframes = {}
loaded_recordsets = []

if len(record_sets) > 0:
    record_set_ids = [rs.id for rs in record_sets]
    for rs_id in record_set_ids:
        print(f"Loading records for RecordSet @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            loaded_recordsets.append(rs_id)
        else:
            print(f"No records found for record set {rs_id}")
    if loaded_recordsets:
        example_rs = loaded_recordsets[0]
        print(f"\nFirst loaded record set @id: {example_rs}\nColumns (these are field @id's):")
        print(dataframes[example_rs].columns.tolist())
        print("\nExample records:")
        display(dataframes[example_rs].head())
    else:
        print("None of the record sets have any records.")
else:
    print("No record sets to extract from.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
All fields referenced by their `@id`.

In the example below, we'll:
- Pick a numeric field (e.g., coverage or MW) for filtering/normalization
- Filter the DataFrame, normalize, and group by a chosen categorical field if available.

In [ ]:
# You may need to adjust the field @ids depending on what's available (see overview)

if loaded_recordsets:
    df = dataframes[example_rs]
    print(f"Analyzing record set: {example_rs}")

    # Try to pick a numeric field
    numeric_candidates = [col for col in df.columns if df[col].dtype in [int, float, np.float64, np.float32, np.int32, np.int64]]
    if len(numeric_candidates) == 0:
        # Try to infer numeric columns (e.g., columns named with 'coverage', 'MW', etc.) as fallback
        for col in df.columns:
            if any(keyword in col.lower() for keyword in ["coverage", "mw", "molecular_weight", "count"]):
                try:
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                except Exception:
                    pass
        numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if df[numeric_field].notna().any() else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} records")
        display(filtered_df[[numeric_field]].head())
        # Normalize field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to find a categorical field for grouping
        group_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped data by {group_field} (showing mean {numeric_field}):")
            display(grouped_df.head())
    else:
        print("No numeric field found for filtering/normalization.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize distribution of the selected numeric field
if loaded_recordsets and numeric_candidates:
    plt.figure(figsize=(7,4))
    df[numeric_field].hist(bins=30, color="cornflowerblue", alpha=0.8)
    plt.title(f"Histogram of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If a group field was found, visualize the top 10 groups
    if group_field:
        grouped = df.groupby(group_field)[numeric_field].mean().sort_values(ascending=False).head(10)
        plt.figure(figsize=(8,5))
        grouped.plot(kind='bar', color='salmon')
        plt.title(f"Top 10 mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- This notebook demonstrated loading, parsing, and exploration of a mass spectrometry protein dataset via `mlcroissant`.
- We referenced all elements (record sets, fields) using their Croissant `@id`.
- Basic EDA was performed, including numeric filtering, normalization, and grouping.
- Refer to the record set and field overviews above for `@id` mappings when developing custom analyses.

Explore the Croissant schema and repeat with other record sets or fields of interest as needed.